# 05 · Live demo: speak Swahili, hear English

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/claverfred/real_time_swahili_english_interpreter/blob/main/notebooks/05_live_demo.ipynb)

Loads the whole pipeline once on the GPU (VAD → Swahili ASR → stabilization → MT → English TTS) and keeps it
in memory, then interprets:

* **your browser microphone** (turn-taking: speak, press *Stop* or wait, then hear the English);
* an **uploaded** audio file, or a file on **Google Drive**;
* a FLEURS recording streamed at **real-time pace** (listener latency, as in a live session).

**Privacy:** recordings are saved only in this runtime and, if Drive is mounted, in
`MyDrive/swahili_interpreter/inputs/` (delete them with the last cell). Nothing is sent to any other service:
every model runs inside this Colab runtime.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is enough; L4/A100/L40S are faster). Optional: add a Hugging Face token as the Colab Secret `HF_TOKEN` (key icon on the left); every default model is public, so it is not required.

Results, converted models and data are kept on Google Drive in `MyDrive/swahili_interpreter/`, so later sessions start faster. Nothing is uploaded anywhere else.

In [ ]:
# @title 1. Setup: clone/update the repository, mount Drive, install, fetch data (safe to re-run)
REPO_URL = "https://github.com/claverfred/real_time_swahili_english_interpreter.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
USE_DRIVE = True  # @param {type:"boolean"}
PROFILE = "colab"  # @param {type:"string"}

import os, subprocess, sys
WORKDIR = "/content" if os.path.isdir("/content") else os.getcwd()
REPO = os.path.join(WORKDIR, REPO_URL.rstrip("/").rsplit("/", 1)[-1].removesuffix(".git"))
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "-1", "--format=commit %h  %cd  %s"], capture_output=True, text=True).stdout)

from src.platform import colab
info = colab.bootstrap(REPO, use_drive=USE_DRIVE)
print("persistent root:", info["root"], "(Google Drive)" if info["on_drive"] else "")
for n in info["notes"]:
    print("NOTE:", n)
print("Hugging Face token:", "from Colab Secrets" if info["hf_login"] else "none (fine: all default models are public)")

r = subprocess.run([sys.executable, "scripts/setup_colab.py", "--profile", PROFILE, "--no-smoke"])
stale = colab.restart_required()
if stale:
    print("Restart the runtime (Runtime > Restart session), then run this cell again:", stale)
assert r.returncode == 0, "setup reported a failure (see the table above)"

In [ ]:
# @title 2. Load the interpreter (models stay on the GPU until the last cell)
colab.preload_cuda_libraries()
from IPython.display import Audio, display
from src.pipeline import InterpreterSession

session = InterpreterSession(PROFILE)
m = session.model_status()
print(f"loaded in {session.load_s} s on {session.interp.runtime.device}: ASR {m['asr']['model']} | MT {m['mt']['model']} | "
      f"TTS {m['tts']['backend']} ({m['tts']['voice']})")
print("GPU memory:", colab.gpu_memory())

def show(r):
    print("Swahili:", r["swahili"] or "(no speech recognised)")
    print("English:", r["english"] or "-")
    for u in r["units"]:
        first = u.get("lat_first_audio_ms")
        print(f"   unit: ASR {u.get('lat_asr_final_ms')} ms, MT {u.get('lat_mt_final_ms')} ms, "
              f"first audio {first} ms after the speaker stopped")
    print(f"input {r['input_s']} s, processed in {r['wall_s']} s" + (" (real-time pace)" if r["realtime"] else ""))
    for e in r["errors"]:
        print("ERROR:", e)
    if len(r["audio"]):
        display(Audio(r["audio"], rate=r["sample_rate"], autoplay=True))

In [ ]:
# @title 3. Browser microphone: run, allow the microphone, speak Swahili, press Stop
MAX_SECONDS = 20  # @param {type:"integer"}
INPUTS = os.path.join(info["root"], "inputs")
wav = colab.record_browser_audio(INPUTS, max_seconds=MAX_SECONDS)
print("recorded", wav)
show(session.interpret(wav))

In [ ]:
# @title 4. Upload an audio file (wav, mp3, m4a, ogg, flac, webm)
for p in colab.upload_audio(os.path.join(info["root"], "inputs")):
    src = p if p.suffix.lower() in (".wav", ".flac", ".ogg") else colab.decode_to_wav(p, p.with_suffix(".wav"))
    print("==", p.name); show(session.interpret(src))

In [ ]:
# @title 5. A file on Google Drive
DRIVE_FILE = "/content/drive/MyDrive/path/to/swahili.wav"  # @param {type:"string"}
show(session.interpret(DRIVE_FILE))

In [ ]:
# @title 6. A FLEURS recording streamed at real-time pace (listener latency)
import json
ITEM = 400  # @param {type:"integer"}
lines = open("data/fleurs_sw/test.jsonl", encoding="utf-8").read().splitlines()
item = json.loads(lines[ITEM])
print("reference:", item["raw"])
show(session.interpret(item["path"], realtime=True))

### Optional: the browser UI through Colab's proxy (experimental)
The full web interface (`serve`) can be opened through Colab's authenticated port proxy. This path has not
been tested on Colab yet; the cells above are the supported demo. It loads a second copy of the models, so run
the last cell first on a T4.

In [ ]:
# @title 7. (Experimental) start the web UI
PORT = 8000
url = colab.proxy_url(PORT)
server = subprocess.Popen([sys.executable, "-m", "src.cli", "serve", "--profile", PROFILE, "--port", str(PORT),
                           "--set", f"server.allowed_origins=[{url}]"],
                          stdout=open("/tmp/swi-serve.log", "w"), stderr=subprocess.STDOUT)
print("starting... open", url, "once /tmp/swi-serve.log says the server is ready")

In [ ]:
# @title 8. Stop: unload the models, free the GPU, optionally delete recordings
DELETE_RECORDINGS = False  # @param {type:"boolean"}
if "server" in globals():
    server.terminate()
summary = session.close()
print("session counters:", summary and summary.get("counters"))
del session
print("GPU memory after cleanup:", colab.free_gpu_memory())
if DELETE_RECORDINGS:
    import shutil
    shutil.rmtree(os.path.join(info["root"], "inputs"), ignore_errors=True)
    print("recordings deleted")